In [1]:
import os
import math
import json
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
import matplotlib.colors
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from datetime import datetime
# from patsy import dmatrices
from patsy import dmatrix
from scipy.stats import ttest_rel
from scipy.stats import ttest_ind
import statsmodels.formula.api as smf
main_path = r'/home/20250114zmz_kd/'

In [2]:
from causalinference import CausalModel
from causalinference.utils import random_data

In [3]:
CIs = {'90': 1.645, '95': 1.96, '99': 2.576}

In [4]:
labels = ['US', 'Others']

In [5]:
data = r'GraduationPaper/RevisetoJournal/9991-MergedData_similarity.csv'
d = pd.read_csv(main_path + data)
del d['Unnamed: 0']
print(d .shape)
d .columns

(317275, 102)


Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'ab_length', 'mean_career_age', 'ex_ld_avg_avgimpact',
       'ex_ld_avg_insthindex', 'ex_ld_avg_before_year_prod_fac',
       'ex_ld_avg_before_year_with_ih', 'ex_ld_avg_before_year_co_lead',
       'ex_ld_avg_before_year_participation', 'knowledge_proximity_mean',
       'knowledge_proximity_max'],
      dtype='object', length=102)

In [6]:
d['CoType'].unique()

array(['Collaboration', 'Service', 'Participation'], dtype=object)

In [7]:
d = d[d['CoType'].isin(['Service','Collaboration'])]
print(d .shape)

(289567, 102)


In [8]:
d['reg_class_bin'] = d.apply(lambda row: 1 if row['CoType'] == 'Collaboration' else 0, axis = 1)
d['reg_class_bin'].value_counts()

reg_class_bin
0    232552
1     57015
Name: count, dtype: int64

In [9]:
# 假设 df 是你的 dataframe
d['PublishedYear'] = d['PublishedYear'].astype('category')

In [10]:
d .columns

Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'mean_career_age', 'ex_ld_avg_avgimpact', 'ex_ld_avg_insthindex',
       'ex_ld_avg_before_year_prod_fac', 'ex_ld_avg_before_year_with_ih',
       'ex_ld_avg_before_year_co_lead', 'ex_ld_avg_before_year_participation',
       'knowledge_proximity_mean', 'knowledge_proximity_max', 'reg_class_bin'],
      dtype='object', length=103)

In [11]:
co_feats_ = ["lnnum_author", "international", "lnnum_reference", "num_fac", "SDG",
             "lnmean_career_age", "lnex_ld_avg_avgimpact", "lnex_ld_avg_insthindex",
             "ex_ld_bin_gs", "ex_ld_bin_sameC", "knowledge_proximity_mean",
             "lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_with_ih_bin",
             'Agricultural and Biological Sciences',
             'Arts and Humanities', 'Biochemistry, Genetics and Molecular Biology',
             'Business, Management and Accounting', 'Chemical Engineering',
             'Chemistry', 'Computer Science', 'Decision Sciences', 'Dentistry',
             'Earth and Planetary Sciences', 'Economics, Econometrics and Finance',
             'Energy', 'Engineering', 'Environmental Science', 'Health Professions',
             'Immunology and Microbiology', 'Materials Science', 'Mathematics',
             'Medicine', 'Neuroscience', 'Nursing','Pharmacology, Toxicology and Pharmaceutics', 'Physics and Astronomy',
             'Psychology', 'Social Sciences', 'Veterinary',]

In [12]:
# 1. 把 DataFrame 里的空格替换成下划线
d.columns = d.columns.str.replace(' ', '_')
d.columns = d.columns.str.replace(',', '')
print(d .columns)
# 2. 把特征列表里的空格也替换掉
co_feats_ = [f.replace(' ', '_') for f in co_feats_]
co_feats_ = [f.replace(',', '') for f in co_feats_]
print(co_feats_)

Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'mean_career_age', 'ex_ld_avg_avgimpact', 'ex_ld_avg_insthindex',
       'ex_ld_avg_before_year_prod_fac', 'ex_ld_avg_before_year_with_ih',
       'ex_ld_avg_before_year_co_lead', 'ex_ld_avg_before_year_participation',
       'knowledge_proximity_mean', 'knowledge_proximity_max', 'reg_class_bin'],
      dtype='object', length=103)
['lnnum_author', 'international', 'lnnum_reference', 'num_fac', 'SDG', 'lnmean_career_age', 'lnex_ld_avg_avgimpact', 'lnex_ld_avg_insthindex', 'ex_ld_bin_gs', 'ex_ld_bin_sameC', 'knowledge_proximity_mean', 'lnex_ld_avg_before_year_prod_fac', 'ex_ld_max_before_year_with_ih_bin', 'Agricultural_and_Biological_Sciences', 'Arts_and_Humanities', 'Biochemistry_Genetics_and_Molecular_Biology', 'Business_Management_and_Accounting', 'Chemical_Engineering', 'Chemistry', 'Comp

In [13]:
X = dmatrix(formula_like=' + '.join(co_feats_), data=d, return_type="dataframe")

In [14]:
list(X.columns)

['Intercept',
 'international[T.international]',
 'SDG[T.True]',
 'ex_ld_bin_gs[T.GlobalSouth]',
 'ex_ld_bin_sameC[T.Same]',
 'ex_ld_max_before_year_with_ih_bin[T.True]',
 'lnnum_author',
 'lnnum_reference',
 'num_fac',
 'lnmean_career_age',
 'lnex_ld_avg_avgimpact',
 'lnex_ld_avg_insthindex',
 'knowledge_proximity_mean',
 'lnex_ld_avg_before_year_prod_fac',
 'Agricultural_and_Biological_Sciences',
 'Arts_and_Humanities',
 'Biochemistry_Genetics_and_Molecular_Biology',
 'Business_Management_and_Accounting',
 'Chemical_Engineering',
 'Chemistry',
 'Computer_Science',
 'Decision_Sciences',
 'Dentistry',
 'Earth_and_Planetary_Sciences',
 'Economics_Econometrics_and_Finance',
 'Energy',
 'Engineering',
 'Environmental_Science',
 'Health_Professions',
 'Immunology_and_Microbiology',
 'Materials_Science',
 'Mathematics',
 'Medicine',
 'Neuroscience',
 'Nursing',
 'Pharmacology_Toxicology_and_Pharmaceutics',
 'Physics_and_Astronomy',
 'Psychology',
 'Social_Sciences',
 'Veterinary']

In [15]:
co_feats = [
 'international[T.international]',
 'SDG[T.True]',
 'ex_ld_bin_gs[T.GlobalSouth]',
 'ex_ld_bin_sameC[T.Same]',
 'ex_ld_max_before_year_with_ih_bin[T.True]',
 'lnnum_author',
 'lnnum_reference',
 'num_fac',
 'lnmean_career_age',
 'lnex_ld_avg_avgimpact',
 'lnex_ld_avg_insthindex',
 'knowledge_proximity_mean',
 'lnex_ld_avg_before_year_prod_fac',
 'Agricultural_and_Biological_Sciences',
 'Arts_and_Humanities',
 'Biochemistry_Genetics_and_Molecular_Biology',
 'Business_Management_and_Accounting',
 'Chemical_Engineering',
 'Chemistry',
 'Computer_Science',
 'Decision_Sciences',
 'Dentistry',
 'Earth_and_Planetary_Sciences',
 'Economics_Econometrics_and_Finance',
 'Energy',
 'Engineering',
 'Environmental_Science',
 'Health_Professions',
 'Immunology_and_Microbiology',
 'Materials_Science',
 'Mathematics',
 'Medicine',
 'Neuroscience',
 'Nursing',
 'Pharmacology_Toxicology_and_Pharmaceutics',
 'Physics_and_Astronomy',
 'Psychology',
 'Social_Sciences',
 'Veterinary'
]

In [16]:
X['CoType'] = d['CoType']
X['reg_class_bin'] = d['reg_class_bin']
X['novel_uzzi_bin'] = d['novel_uzzi_bin']

In [17]:
# Y is the outcome, D is treatment status, and X is the independent variable
causal = CausalModel(Y=X['novel_uzzi_bin'].values, D=X['reg_class_bin'].values, \
                     X=X[co_feats].values)

In [18]:
print(causal.summary_stats)


Summary Statistics

                    Controls (N_c=232552)       Treated (N_t=57015)             
       Variable         Mean         S.d.         Mean         S.d.     Raw-diff
--------------------------------------------------------------------------------
              Y        0.373        0.484        0.375        0.484        0.002

                    Controls (N_c=232552)       Treated (N_t=57015)             
       Variable         Mean         S.d.         Mean         S.d.     Nor-diff
--------------------------------------------------------------------------------
             X0        0.439        0.496        0.729        0.445        0.616
             X1        0.486        0.500        0.452        0.498       -0.068
             X2        0.063        0.243        0.060        0.238       -0.011
             X3        0.531        0.499        0.293        0.455       -0.500
             X4        0.681        0.466        0.879        0.326        0.492
      

In [19]:
causal.est_propensity()

In [20]:
# Propensity model results
print(causal.propensity)


Estimated Parameters of Propensity Score

                    Coef.       S.e.          z      P>|z|      [95% Conf. int.]
--------------------------------------------------------------------------------
     Intercept     -2.496      0.084    -29.872      0.000     -2.660     -2.332
            X0      0.767      0.012     62.646      0.000      0.743      0.791
            X1     -0.012      0.010     -1.163      0.245     -0.033      0.008
            X2     -0.432      0.022    -20.085      0.000     -0.474     -0.390
            X3     -0.502      0.012    -41.646      0.000     -0.526     -0.478
            X4      1.212      0.017     73.397      0.000      1.180      1.245
            X5      0.197      0.010     19.430      0.000      0.177      0.217
            X6     -0.035      0.010     -3.659      0.000     -0.054     -0.016
            X7      0.644      0.008     80.180      0.000      0.628      0.660
            X8      0.485      0.017     28.409      0.000      0.

In [21]:
causal.propensity['fitted']

array([0.34908025, 0.25125813, 0.11840824, ..., 0.34608234, 0.12241704,
       0.12141247])

In [22]:
d['CoType'].unique()

array(['Collaboration', 'Service'], dtype=object)

In [23]:
for feat in co_feats:
    print(feat)
    af_avg = np.mean(X.loc[X['CoType']=='Collaboration', feat])
    nam_avg = np.mean(X.loc[X['CoType']=='Service', feat])
    print('\tCollaboration:\t', af_avg)
    print('\tService:\t', nam_avg)
    print('\tDiff:\t', af_avg-nam_avg)
    print('\tT-test:\t', ttest_ind(X.loc[X['CoType']=='Collaboration', feat], X.loc[X['CoType']=='Service', feat])[1])

international[T.international]
	Collaboration:	 0.7288432868543365
	Service:	 0.43872338229729263
	Diff:	 0.2901199045570439
	T-test:	 0.0
SDG[T.True]
	Collaboration:	 0.4518109269490485
	Service:	 0.485650521173759
	Diff:	 -0.0338395942247105
	T-test:	 1.2546839854538284e-47
ex_ld_bin_gs[T.GlobalSouth]
	Collaboration:	 0.06045777426992897
	Service:	 0.06303536413361313
	Diff:	 -0.002577589863684164
	T-test:	 0.02271897703264723
ex_ld_bin_sameC[T.Same]
	Collaboration:	 0.2925195124090152
	Service:	 0.5312059238363893
	Diff:	 -0.23868641142737412
	T-test:	 0.0
ex_ld_max_before_year_with_ih_bin[T.True]
	Collaboration:	 0.8789441375076734
	Service:	 0.6808369740961161
	Diff:	 0.19810716341155732
	T-test:	 0.0
lnnum_author
	Collaboration:	 2.154429482909093
	Service:	 1.9980434311474133
	Diff:	 0.15638605176167975
	T-test:	 0.0
lnnum_reference
	Collaboration:	 3.6086096222351873
	Service:	 3.6534719171420424
	Diff:	 -0.044862294906855116
	T-test:	 2.2457645558053295e-61
num_fac
	Collaborat

In [24]:
len(causal.propensity['fitted'])

289567

In [25]:
X['pscore'] = causal.propensity['fitted']

In [26]:
tem = X[['CoType', 'pscore']].sort_values(by = ['pscore'])
tem['index'] = tem.index

In [27]:
tem = tem.values.tolist()

In [28]:
tem[-10:]

[['Collaboration', 0.9640495298573696, 116581],
 ['Collaboration', 0.9643910769236173, 154647],
 ['Collaboration', 0.9644278487141538, 154648],
 ['Collaboration', 0.9650244239344528, 154650],
 ['Collaboration', 0.9666085237639177, 209590],
 ['Collaboration', 0.966657931401243, 209591],
 ['Collaboration', 0.9667245271835365, 209586],
 ['Collaboration', 0.9667743204986574, 209589],
 ['Collaboration', 0.9667771875186325, 209587],
 ['Collaboration', 0.966826195691196, 209588]]

In [29]:
leng = len(tem)

In [30]:
print(leng)

289567


In [31]:
pairs = {}
for i, elms in enumerate(tem):
    gen, score, ix = elms
    if gen == 'Collaboration':
        pre_nam_ix, nex_nam_ix = 0, 0
        j = i-1
        while j >= 0:
            if tem[j][0] != 'Service':
                j -= 1
            else:
                break
        if j >= 0:
            pre_nam_ix = j
        n = i+1
        while n <= leng-1:
            if tem[n][0] != 'Service':
                n += 1
            else:
                break
        if n <= leng-1:
            nex_nam_ix = n
        if abs(score - tem[pre_nam_ix][1]) <= abs(score - tem[nex_nam_ix][1]):
            pairs[ix] = tem[pre_nam_ix][2]
        else:
            pairs[ix] = tem[nex_nam_ix][2]

In [32]:
len(pairs)

57015

In [33]:
for feat in co_feats:
    print('\n')
    print('Feat:', feat, '\n')
    print('Before matching:\n')
    af_avg = np.mean(X.loc[X['CoType']=='Collaboration', feat])
    nam_avg = np.mean(X.loc[X['CoType']=='Service', feat])
    print('\tCollaboration:\t', af_avg)
    print('\tService:\t', nam_avg)
    print('\tDiff:\t', af_avg-nam_avg)
    print('\tT-test:\t', ttest_ind(X.loc[X['CoType']=='Collaboration', feat], X.loc[X['CoType']=='Service', feat])[1])

    print('\nAfter matching:\n')
    af_avg = np.mean(X.loc[pairs.keys(), feat])
    nam_avg = np.mean(X.loc[pairs.values(), feat])
    print('\tCollaboration:\t', af_avg)
    print('\tService:\t', nam_avg)
    print('\tDiff:\t', af_avg-nam_avg)
    print('\tT-test:\t', ttest_rel(X.loc[pairs.keys(), feat], X.loc[pairs.values(), feat])[1])



Feat: international[T.international] 

Before matching:

	Collaboration:	 0.7288432868543365
	Service:	 0.43872338229729263
	Diff:	 0.2901199045570439
	T-test:	 0.0

After matching:

	Collaboration:	 0.7288432868543365
	Service:	 0.721371568885381
	Diff:	 0.007471717968955516
	T-test:	 0.0003444444816038998


Feat: SDG[T.True] 

Before matching:

	Collaboration:	 0.4518109269490485
	Service:	 0.485650521173759
	Diff:	 -0.0338395942247105
	T-test:	 1.2546839854538284e-47

After matching:

	Collaboration:	 0.4518109269490485
	Service:	 0.4553012365167061
	Diff:	 -0.0034903095676576124
	T-test:	 0.23711254225168127


Feat: ex_ld_bin_gs[T.GlobalSouth] 

Before matching:

	Collaboration:	 0.06045777426992897
	Service:	 0.06303536413361313
	Diff:	 -0.002577589863684164
	T-test:	 0.02271897703264723

After matching:

	Collaboration:	 0.06045777426992897
	Service:	 0.05719547487503289
	Diff:	 0.0032622993948960774
	T-test:	 0.0185969066376039


Feat: ex_ld_bin_sameC[T.Same] 

Before matching

In [34]:
for feat in co_feats:
    af_avg = np.mean(X.loc[X['CoType']=='Collaboration', feat])
    nam_avg = np.mean(X.loc[X['CoType']=='Service', feat])
    af_avg_ = np.mean(X.loc[pairs.keys(), feat])
    nam_avg_ = np.mean(X.loc[pairs.values(), feat])
    print(feat, ' & ', '{:6.2f}'.format(af_avg), ' & ', '{:6.2f}'.format(nam_avg), ' & ', '{:6.2f}'.format(af_avg_), ' & ', '{:6.2f}'.format(nam_avg_), ' \\\\ \hline')

international[T.international]  &    0.73  &    0.44  &    0.73  &    0.72  \\ \hline
SDG[T.True]  &    0.45  &    0.49  &    0.45  &    0.46  \\ \hline
ex_ld_bin_gs[T.GlobalSouth]  &    0.06  &    0.06  &    0.06  &    0.06  \\ \hline
ex_ld_bin_sameC[T.Same]  &    0.29  &    0.53  &    0.29  &    0.30  \\ \hline
ex_ld_max_before_year_with_ih_bin[T.True]  &    0.88  &    0.68  &    0.88  &    0.88  \\ \hline
lnnum_author  &    2.15  &    2.00  &    2.15  &    2.17  \\ \hline
lnnum_reference  &    3.61  &    3.65  &    3.61  &    3.61  \\ \hline
num_fac  &    1.53  &    1.24  &    1.53  &    1.51  \\ \hline
lnmean_career_age  &    3.14  &    3.09  &    3.14  &    3.14  \\ \hline
lnex_ld_avg_avgimpact  &    2.92  &    3.00  &    2.92  &    2.93  \\ \hline
lnex_ld_avg_insthindex  &    5.96  &    6.10  &    5.96  &    5.96  \\ \hline
knowledge_proximity_mean  &    0.74  &    0.76  &    0.74  &    0.74  \\ \hline
lnex_ld_avg_before_year_prod_fac  &    2.24  &    2.19  &    2.24  &    2.21  

In [35]:
# AS avg
np.mean(X.loc[pairs.keys(), 'novel_uzzi_bin'])

np.float64(0.3749890379724634)

In [36]:
# NAM avg
np.mean(X.loc[pairs.values(), 'novel_uzzi_bin'])

np.float64(0.35632728229413313)

In [37]:
ttest_rel(X.loc[pairs.keys(), 'novel_uzzi_bin'].apply(lambda x: 1 if x == True else 0), \
          X.loc[pairs.values(), 'novel_uzzi_bin'].apply(lambda x: 1 if x == True else 0))

TtestResult(statistic=np.float64(6.551122234729801), pvalue=np.float64(5.75905922383217e-11), df=np.int64(57014))

In [38]:
y_treated = X.loc[list(pairs.keys()), 'novel_uzzi_bin'].values
y_control = X.loc[list(pairs.values()), 'novel_uzzi_bin'].values

n_boot = 1000  # bootstrap次数
att_boot = np.zeros(n_boot)
n = len(y_treated)

for i in range(n_boot):
    idx = np.random.randint(0, n, n)  # 随机抽样索引，有放回
    att_boot[i] = np.mean(y_treated[idx] - y_control[idx])

# ATT估计值
att = np.mean(y_treated - y_control)

# 95%置信区间
ci_lower = np.percentile(att_boot, 2.5)
ci_upper = np.percentile(att_boot, 97.5)

print("ATT:", att)
print("95% CI: [{:.4f}, {:.4f}]".format(ci_lower, ci_upper))

ATT: 0.018661755678330266
95% CI: [0.0133, 0.0241]


In [39]:
d['novel_uzzi_bin'] = d['novel_uzzi_bin'].astype('int32')

In [40]:
print(d .shape)

(289567, 103)


In [41]:
dpsm = d.loc[list(pairs.keys()) + list(pairs.values())].sample(frac=1)
print(dpsm .shape)

(114030, 103)


In [42]:
dpsm .columns

Index(['work_id', 'PublishedYear', 'Facility', 'num_fac', 'paper_type',
       'paper_language', 'novel_uzzi', 'novel_uzzi_bin', 'num_fac_scientist',
       'ratio_fac_scientist',
       ...
       'mean_career_age', 'ex_ld_avg_avgimpact', 'ex_ld_avg_insthindex',
       'ex_ld_avg_before_year_prod_fac', 'ex_ld_avg_before_year_with_ih',
       'ex_ld_avg_before_year_co_lead', 'ex_ld_avg_before_year_participation',
       'knowledge_proximity_mean', 'knowledge_proximity_max', 'reg_class_bin'],
      dtype='object', length=103)

In [43]:
dpsm['work_id'].nunique()

41165

In [52]:
Update

NameError: name 'Update' is not defined

In [44]:
dpsm .to_csv(main_path + r'science_media_coverage/260128Revision/PSM-FirstCorresponding/PSM-sample-260311-africa.csv')